<a href="https://www.kaggle.com/code/pyrisforge/easy-realesrgan-image-upscaler?scriptVersionId=318735722" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# **🚀 Easy-RealESRGAN-Image-Upscaler**

# 📦 1. Environment Setup & Initialization

In [1]:
# ==============================================================================
# EASY-REALESRGAN ENVIRONMENT INITIALIZATION
# Repository Setup + Dependency Installation + Python Compatibility Patch
# Optimized for Kaggle / Colab Runtime
# ==============================================================================

import gc
import logging
import os
import platform
import subprocess
import sys
import importlib.util
from pathlib import Path
from datetime import datetime


# ==============================================================================
# LOGGING SYSTEM INITIALIZATION
# ==============================================================================

LOG_DIR = "logs"
os.makedirs(LOG_DIR, exist_ok=True)

LOG_FILE = os.path.join(LOG_DIR, "latest.log")
ERROR_LOG_FILE = os.path.join(LOG_DIR, "errors.log")

logger = logging.getLogger("EasyRealESRGAN")
logger.setLevel(logging.INFO)
logger.handlers.clear()

formatter = logging.Formatter(
    "[%(asctime)s][%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

console_handler = logging.StreamHandler()
console_handler.setFormatter(formatter)

file_handler = logging.FileHandler(LOG_FILE)
file_handler.setFormatter(formatter)

error_handler = logging.FileHandler(ERROR_LOG_FILE)
error_handler.setLevel(logging.ERROR)
error_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)
logger.addHandler(error_handler)

logger.info("Logging system initialized")


# ==============================================================================
# GLOBAL CONFIGURATION
# ==============================================================================

CONFIG = {
    "project_name": "Real-ESRGAN",
    "repo_url": "https://github.com/xinntao/Real-ESRGAN.git",
    "enable_memory_cleanup": True,
    "save_logs": True,
    "use_fp16": True,
}


# ==============================================================================
# RUNTIME INFORMATION
# ==============================================================================

def print_runtime_info():
    logger.info("Collecting runtime environment information...")

    logger.info(f"Python Version : {platform.python_version()}")
    logger.info(f"Platform       : {platform.system()} {platform.release()}")

    try:
        import torch

        logger.info(f"PyTorch Version: {torch.__version__}")
        logger.info(f"CUDA Available : {torch.cuda.is_available()}")

        if torch.cuda.is_available():
            gpu_name = torch.cuda.get_device_name(0)
            total_vram = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)

            logger.info(f"GPU Name       : {gpu_name}")
            logger.info(f"Total VRAM     : {total_vram} GB")

    except Exception:
        logger.exception("Failed to collect PyTorch runtime information")


# ==============================================================================
# ENVIRONMENT INITIALIZATION
# ==============================================================================

def initialize_environment():
    logger.info("=" * 60)
    logger.info("STARTING EASY-REALESRGAN ENVIRONMENT SETUP")
    logger.info("=" * 60)

    project_name = CONFIG["project_name"]
    project_dir = Path(project_name)
    current_dir = Path.cwd()

    try:
        # ----------------------------------------------------------------------
        # STEP 1: REPOSITORY SETUP
        # ----------------------------------------------------------------------
        logger.info("[1/4] Checking repository directory...")

        if current_dir.name == project_name:
            logger.info(f"Already inside '{project_name}' directory")

        elif project_dir.exists():
            logger.info(f"Repository folder found: '{project_name}'")
            os.chdir(project_dir)

        else:
            logger.info(f"Cloning repository: {project_name}")

            subprocess.run(
                [
                    "git",
                    "clone",
                    CONFIG["repo_url"],
                    "-q"
                ],
                check=True
            )

            os.chdir(project_dir)

        logger.info(f"Current working directory: {Path.cwd()}")


        # ----------------------------------------------------------------------
        # STEP 2: INSTALL DEPENDENCIES
        # ----------------------------------------------------------------------
        logger.info("[2/4] Installing dependencies...")

        pip_cmd = [sys.executable, "-m", "pip", "install", "-q"]

        install_commands = [
            ["-r", "requirements.txt"],
            ["gfpgan", "facexlib", "gdown"],
            ["-e", "."]
        ]

        for command in install_commands:
            logger.info(f"Running pip install: {' '.join(command)}")
            subprocess.run(pip_cmd + command, check=True)


        # ----------------------------------------------------------------------
        # STEP 3: REGISTER PROJECT PATH
        # ----------------------------------------------------------------------
        logger.info("[3/4] Registering project path...")

        current_path = str(Path.cwd())

        if current_path not in sys.path:
            sys.path.append(current_path)
            logger.info("Project path added to sys.path")


        # ----------------------------------------------------------------------
        # STEP 4: PYTHON 3.12+ COMPATIBILITY PATCH
        # ----------------------------------------------------------------------
        logger.info("[4/4] Checking basicsr compatibility patch...")

        spec = importlib.util.find_spec("basicsr")

        if not (spec and spec.origin):
            logger.warning("Library 'basicsr' not found. Patch skipped.")
            return

        basicsr_path = Path(spec.origin).parent
        target_file = basicsr_path / "data" / "degradations.py"

        if not target_file.exists():
            logger.error(f"Patch target file not found: {target_file}")
            return

        content = target_file.read_text(encoding="utf-8")

        old_import = (
            "from torchvision.transforms.functional_tensor "
            "import rgb_to_grayscale"
        )

        new_import = (
            "from torchvision.transforms.functional "
            "import rgb_to_grayscale"
        )

        if old_import in content:
            content = content.replace(old_import, new_import)
            target_file.write_text(content, encoding="utf-8")

            logger.info(
                "Compatibility patch applied successfully "
                "(functional_tensor → functional)"
            )

        else:
            logger.info("Compatibility patch already applied or not required")


        # ----------------------------------------------------------------------
        # FINALIZATION
        # ----------------------------------------------------------------------
        print_runtime_info()

        logger.info("Environment setup completed successfully")


        if CONFIG["enable_memory_cleanup"]:
            try:
                import torch

                gc.collect()
                torch.cuda.empty_cache()

                logger.info("Memory cleanup completed")

            except Exception:
                logger.exception("Memory cleanup failed")


    except subprocess.CalledProcessError as error:
        logger.exception(
            f"Dependency installation process failed: {error}"
        )

    except Exception:
        logger.exception("Fatal error during environment initialization")


# ==============================================================================
# EXECUTION
# ==============================================================================

if __name__ == "__main__":
    initialize_environment()

[2026-05-12 16:24:05][INFO] Logging system initialized
[2026-05-12 16:24:05][INFO] ============================================================
[2026-05-12 16:24:05][INFO] STARTING EASY-REALESRGAN ENVIRONMENT SETUP
[2026-05-12 16:24:05][INFO] ============================================================
[2026-05-12 16:24:05][INFO] [1/4] Checking repository directory...
[2026-05-12 16:24:05][INFO] Cloning repository: Real-ESRGAN
[2026-05-12 16:24:06][INFO] Current working directory: /kaggle/working/Real-ESRGAN
[2026-05-12 16:24:06][INFO] [2/4] Installing dependencies...
[2026-05-12 16:24:06][INFO] Running pip install: -r requirements.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 17.1 MB/s eta 0:00:00


[2026-05-12 16:24:18][INFO] Running pip install: gfpgan facexlib gdown
[2026-05-12 16:24:22][INFO] Running pip install: -e .
[2026-05-12 16:24:29][INFO] [3/4] Registering project path...
[2026-05-12 16:24:29][INFO] Project path added to sys.path
[2026-05-12 16:24:29][INFO] [4/4] Checking basicsr compatibility patch...
[2026-05-12 16:24:29][INFO] Compatibility patch applied successfully (functional_tensor → functional)
[2026-05-12 16:24:29][INFO] Collecting runtime environment information...
[2026-05-12 16:24:29][INFO] Python Version : 3.12.12
[2026-05-12 16:24:29][INFO] Platform       : Linux 6.6.122+
[2026-05-12 16:24:33][INFO] PyTorch Version: 2.8.0+cu126
[2026-05-12 16:24:33][INFO] CUDA Available : True
[2026-05-12 16:24:33][INFO] GPU Name       : Tesla T4
[2026-05-12 16:24:33][INFO] Total VRAM     : 14.56 GB
[2026-05-12 16:24:33][INFO] Environment setup completed successfully
[2026-05-12 16:24:33][INFO] Memory cleanup completed


# 📥 2. Smart Model Downloader

In [2]:
# ==============================================================================
# SMART MODEL DOWNLOADER
# Download + Validation + Retry + Progress Tracking
# Optimized for Kaggle / Colab Runtime
# ==============================================================================

import time
import requests
from pathlib import Path
from tqdm.auto import tqdm


# ==============================================================================
# MODEL CONFIGURATION
# ==============================================================================

WEIGHTS_DIR = Path("weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_REGISTRY = {
    "RealESRGAN_x4plus.pth": {
        "url": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"
    },

    "realesr-animevideov3.pth": {
        "url": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth"
    },

    "RealESRGAN_x2plus.pth": {
        "url": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth"
    },
}


# ==============================================================================
# HELPER FUNCTIONS
# ==============================================================================

def get_remote_file_size(url: str) -> int:
    """
    Retrieve remote file size using HEAD request.
    """

    try:
        response = requests.head(
            url,
            allow_redirects=True,
            timeout=15
        )

        response.raise_for_status()

        return int(response.headers.get("content-length", 0))

    except Exception:
        logger.exception(f"Failed to retrieve remote file size: {url}")
        return 0


def validate_existing_file(file_path: Path, remote_size: int) -> bool:
    """
    Validate existing local file using file size comparison.
    """

    if not file_path.exists():
        return False

    local_size = file_path.stat().st_size

    if remote_size > 0 and local_size == remote_size:
        logger.info(
            f"Valid model already exists: "
            f"{file_path.name} "
            f"({local_size / (1024 ** 2):.2f} MB)"
        )
        return True

    logger.warning(
        f"File validation mismatch detected: "
        f"{file_path.name} "
        f"(Local: {local_size} | Remote: {remote_size})"
    )

    return False


# ==============================================================================
# DOWNLOAD FUNCTION
# ==============================================================================

def download_model(
    url: str,
    destination: Path,
    max_retries: int = 3
) -> bool:
    """
    Download model with:
    - retry handling
    - smart validation
    - progress tracking
    - integrity verification
    """

    filename = destination.name

    logger.info(f"Preparing model download: {filename}")

    remote_size = get_remote_file_size(url)

    if validate_existing_file(destination, remote_size):
        logger.info(f"Skipping download: {filename}")
        return True

    destination.parent.mkdir(parents=True, exist_ok=True)

    for attempt in range(1, max_retries + 1):

        try:
            logger.info(
                f"Downloading {filename} "
                f"(Attempt {attempt}/{max_retries})"
            )

            response = requests.get(
                url,
                stream=True,
                timeout=60
            )

            response.raise_for_status()

            total_size = int(
                response.headers.get("content-length", 0)
            )

            start_time = time.time()

            with destination.open("wb") as file, tqdm(
                desc=filename,
                total=total_size,
                unit="iB",
                unit_scale=True,
                unit_divisor=1024,
            ) as progress_bar:

                for chunk in response.iter_content(chunk_size=8192):

                    if chunk:
                        written = file.write(chunk)
                        progress_bar.update(written)

            elapsed_time = time.time() - start_time

            downloaded_size = destination.stat().st_size

            if remote_size > 0 and downloaded_size != remote_size:

                logger.error(
                    f"Downloaded file validation failed: {filename}"
                )

                continue

            logger.info(
                f"Download completed successfully: "
                f"{filename} "
                f"({downloaded_size / (1024 ** 2):.2f} MB | "
                f"{elapsed_time:.2f}s)"
            )

            return True

        except requests.exceptions.RequestException:
            logger.exception(
                f"Network error while downloading: {filename}"
            )

        except Exception:
            logger.exception(
                f"Unexpected error during download: {filename}"
            )

        if attempt < max_retries:
            logger.warning(
                f"Retrying download in 3 seconds: {filename}"
            )
            time.sleep(3)

    logger.error(f"Failed to download model: {filename}")

    return False


# ==============================================================================
# MODEL SETUP
# ==============================================================================

def setup_models():

    logger.info("=" * 60)
    logger.info("STARTING MODEL VALIDATION & DOWNLOAD")
    logger.info("=" * 60)

    successful_downloads = 0
    failed_downloads = []

    for model_name, model_info in MODEL_REGISTRY.items():

        model_path = WEIGHTS_DIR / model_name

        success = download_model(
            url=model_info["url"],
            destination=model_path,
            max_retries=3
        )

        if success:
            successful_downloads += 1

        else:
            failed_downloads.append(model_name)

    logger.info("=" * 60)
    logger.info("MODEL SETUP SUMMARY")
    logger.info("=" * 60)

    logger.info(f"Successful Downloads : {successful_downloads}")
    logger.info(f"Failed Downloads     : {len(failed_downloads)}")

    if failed_downloads:
        logger.warning(
            f"Failed Models: {', '.join(failed_downloads)}"
        )

    logger.info("Model setup completed")


# ==============================================================================
# EXECUTION
# ==============================================================================

if __name__ == "__main__":
    setup_models()

[2026-05-12 16:24:33][INFO] ============================================================
[2026-05-12 16:24:33][INFO] STARTING MODEL VALIDATION & DOWNLOAD
[2026-05-12 16:24:33][INFO] ============================================================
[2026-05-12 16:24:33][INFO] Preparing model download: RealESRGAN_x4plus.pth
[2026-05-12 16:24:34][INFO] Downloading RealESRGAN_x4plus.pth (Attempt 1/3)


RealESRGAN_x4plus.pth:   0%|          | 0.00/63.9M [00:00<?, ?iB/s]

[2026-05-12 16:24:35][INFO] Download completed successfully: RealESRGAN_x4plus.pth (63.94 MB | 0.91s)
[2026-05-12 16:24:35][INFO] Preparing model download: realesr-animevideov3.pth
[2026-05-12 16:24:35][INFO] Downloading realesr-animevideov3.pth (Attempt 1/3)


realesr-animevideov3.pth:   0%|          | 0.00/2.39M [00:00<?, ?iB/s]

[2026-05-12 16:24:36][INFO] Download completed successfully: realesr-animevideov3.pth (2.39 MB | 0.19s)
[2026-05-12 16:24:36][INFO] Preparing model download: RealESRGAN_x2plus.pth
[2026-05-12 16:24:36][INFO] Downloading RealESRGAN_x2plus.pth (Attempt 1/3)


RealESRGAN_x2plus.pth:   0%|          | 0.00/64.0M [00:00<?, ?iB/s]

[2026-05-12 16:24:37][INFO] Download completed successfully: RealESRGAN_x2plus.pth (63.96 MB | 0.44s)
[2026-05-12 16:24:37][INFO] ============================================================
[2026-05-12 16:24:37][INFO] MODEL SETUP SUMMARY
[2026-05-12 16:24:37][INFO] ============================================================
[2026-05-12 16:24:37][INFO] Successful Downloads : 3
[2026-05-12 16:24:37][INFO] Failed Downloads     : 0
[2026-05-12 16:24:37][INFO] Model setup completed


# 🧠 3. Core Engine & Configuration

In [3]:
# ==============================================================================
# CORE ENGINE & CONFIGURATION
# Real-ESRGAN Inference Engine + Adaptive Processing System
# Optimized for Kaggle / Colab Runtime
# ==============================================================================

import gc
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Optional, Tuple

import cv2
import torch


logger.info("=" * 60)
logger.info("INITIALIZING CORE ENGINE")
logger.info("=" * 60)


# ==============================================================================
# QUALITY CONFIGURATION
# ==============================================================================

class QualityConfig:
    """
    Centralized model and quality preset configuration.
    """

    MODELS = {

        "general": {
            "x4": (
                "RealESRGAN_x4plus.pth",
                "RealESRGAN_x4plus"
            ),

            "x2": (
                "RealESRGAN_x2plus.pth",
                "RealESRGAN_x2plus"
            ),
        },

        "anime": {
            "x4": (
                "realesr-animevideov3.pth",
                "realesr-animevideov3"
            ),

            "x4_alt": (
                "RealESRGAN_x4plus_anime_6B.pth",
                "RealESRGAN_x4plus_anime_6B"
            ),

            "x2": (
                "RealESRGAN_x2plus.pth",
                "RealESRGAN_x2plus"
            ),
        },
    }

    # --------------------------------------------------------------------------
    # QUALITY PRESETS
    # --------------------------------------------------------------------------

    PRESETS = {

        # Best quality / Highest VRAM usage
        "ultra": {
            "fp32": True,
            "tile": 0,
            "face_enhance": True,
        },

        # Recommended for T4 / medium GPU
        "high": {
            "fp32": True,
            "tile": 200,
            "face_enhance": False,
        },

        # Recommended default
        "balanced": {
            "fp32": False,
            "tile": 400,
            "face_enhance": False,
        },

        # Fastest / Lowest VRAM usage
        "fast": {
            "fp32": False,
            "tile": 600,
            "face_enhance": False,
        },
    }

    # --------------------------------------------------------------------------
    # ADAPTIVE TILE FALLBACK
    # --------------------------------------------------------------------------

    TILE_CANDIDATES = [600, 400, 200, 100]


# ==============================================================================
# CORE UPSCALER ENGINE
# ==============================================================================

class RealESRGANUpscaler:

    def __init__(self):

        self.input_dir = Path("inputs_temp")
        self.output_dir = Path("results_temp")
        self.weights_dir = Path("weights")

        self.input_dir.mkdir(exist_ok=True)
        self.output_dir.mkdir(exist_ok=True)

        logger.info("Temporary directories initialized")


    # ==========================================================================
    # TEMP DIRECTORY CLEANUP
    # ==========================================================================

    def clean_temp_dirs(self):

        for folder in [self.input_dir, self.output_dir]:

            for item in folder.iterdir():

                try:
                    if item.is_file():
                        item.unlink()

                except Exception:
                    logger.exception(
                        f"Failed to clean temporary file: {item}"
                    )


    # ==========================================================================
    # MODEL RESOLUTION
    # ==========================================================================

    def resolve_model(
        self,
        image_type: str,
        scale: int
    ):

        model_key = "x4" if scale == 4 else "x2"

        if image_type == "anime":

            target_model = (
                QualityConfig.MODELS["anime"][model_key][0]
            )

            if not (self.weights_dir / target_model).exists():

                logger.warning(
                    "Primary anime model not found. "
                    "Using fallback anime model."
                )

                model_key = "x4_alt"

            return QualityConfig.MODELS["anime"][model_key]

        return QualityConfig.MODELS["general"][model_key]


    # ==========================================================================
    # SUBPROCESS EXECUTION
    # ==========================================================================

    def execute_inference(
        self,
        model_name: str,
        scale: int,
        tile_size: int,
        preset: dict,
        timeout: int = 300
    ) -> bool:

        cmd = [
            sys.executable,
            "inference_realesrgan.py",

            "-i",
            str(self.input_dir),

            "-o",
            str(self.output_dir),

            "-n",
            model_name,

            "-s",
            str(scale),

            "-t",
            str(tile_size),
        ]

        if preset["fp32"]:
            cmd.append("--fp32")

        if preset["face_enhance"]:
            cmd.append("--face_enhance")

        logger.info(
            f"Engine Configuration | "
            f"Model: {model_name} | "
            f"Tile: {tile_size} | "
            f"FP32: {preset['fp32']}"
        )

        try:

            start_time = time.time()

            result = subprocess.run(
                cmd,
                capture_output=True,
                text=True,
                timeout=timeout
            )

            elapsed_time = time.time() - start_time

            if result.returncode != 0:

                logger.error(
                    f"Inference failed:\n{result.stderr}"
                )

                return False

            logger.info(
                f"Inference completed successfully "
                f"({elapsed_time:.2f}s)"
            )

            return True

        except subprocess.TimeoutExpired:

            logger.error(
                "Inference timeout exceeded "
                f"({timeout} seconds)"
            )

            return False

        except Exception:

            logger.exception(
                "Unexpected error during inference execution"
            )

            return False


    # ==========================================================================
    # MAIN UPSCALE PROCESS
    # ==========================================================================

    def run(
        self,
        image_path: str | Path,
        image_type: str = "general",
        scale: int = 4,
        quality: str = "balanced",
        output_name: str = None
    ) -> Optional[Tuple[Path, str]]:

        image_path = Path(image_path)

        logger.info(f"Starting upscale process: {image_path.name}")

        self.clean_temp_dirs()

        temp_input = self.input_dir / image_path.name

        shutil.copy(image_path, temp_input)

        preset = QualityConfig.PRESETS.get(
            quality,
            QualityConfig.PRESETS["balanced"]
        )

        model_file, model_name = self.resolve_model(
            image_type=image_type,
            scale=scale
        )

        # ----------------------------------------------------------------------
        # ADAPTIVE TILE FALLBACK SYSTEM
        # ----------------------------------------------------------------------

        tile_candidates = [
            preset["tile"]
        ] + [
            t for t in QualityConfig.TILE_CANDIDATES
            if t != preset["tile"]
        ]

        success = False

        for tile_size in tile_candidates:

            logger.info(
                f"Attempting inference with tile size: {tile_size}"
            )

            success = self.execute_inference(
                model_name=model_name,
                scale=scale,
                tile_size=tile_size,
                preset=preset
            )

            if success:
                break

            logger.warning(
                f"Inference failed using tile size: {tile_size}"
            )

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            gc.collect()

        if not success:

            logger.error(
                f"All inference attempts failed: {image_path.name}"
            )

            return None

        output_files = list(self.output_dir.iterdir())

        if not output_files:

            logger.error(
                "Inference completed but no output file was generated"
            )

            return None

        temp_result = output_files[0]

        final_stem = (
            output_name
            if output_name
            else temp_result.stem
        )

        final_name = f"{final_stem}{temp_result.suffix}"

        logger.info(
            f"Upscale process completed: {final_name}"
        )

        return temp_result, final_name


# ==============================================================================
# GLOBAL ENGINE INSTANCE
# ==============================================================================

upscaler_engine = RealESRGANUpscaler()


# ==============================================================================
# MAIN UPSCALE FUNCTION
# ==============================================================================

def upscale_image(
    image_path: str | Path,
    output_folder: str | Path,
    **kwargs
) -> Optional[Path]:

    image_path = Path(image_path)
    output_folder = Path(output_folder)

    output_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    logger.info(f"Processing image: {image_path.name}")

    try:

        result = upscaler_engine.run(
            image_path=image_path,
            **kwargs
        )

        if not result:

            logger.error(
                "Upscale failed: no output generated"
            )

            return None

        temp_path, final_name = result

        final_path = output_folder / final_name

        # ----------------------------------------------------------------------
        # AUTO FILE RENAME
        # ----------------------------------------------------------------------

        counter = 1

        while final_path.exists():

            final_path = (
                output_folder /
                f"{Path(final_name).stem}_{counter}"
                f"{Path(final_name).suffix}"
            )

            counter += 1

        shutil.move(
            str(temp_path),
            str(final_path)
        )

        # ----------------------------------------------------------------------
        # OUTPUT VALIDATION
        # ----------------------------------------------------------------------

        image = cv2.imread(str(final_path))

        if image is not None:

            height, width = image.shape[:2]

            logger.info(
                f"Upscale successful: "
                f"{final_path.name} "
                f"({width}x{height}px)"
            )

        else:

            logger.warning(
                f"Upscale completed but resolution check failed: "
                f"{final_path.name}"
            )

        return final_path

    except Exception:

        logger.exception(
            f"Exception while processing image: "
            f"{image_path.name}"
        )

        return None

    finally:

        # ----------------------------------------------------------------------
        # MEMORY CLEANUP
        # ----------------------------------------------------------------------

        try:

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            logger.info(
                "Memory cleanup completed"
            )

        except Exception:

            logger.exception(
                "Memory cleanup failed"
            )


logger.info(
    "Core engine initialized successfully"
)

[2026-05-12 16:24:37][INFO] ============================================================
[2026-05-12 16:24:37][INFO] INITIALIZING CORE ENGINE
[2026-05-12 16:24:37][INFO] ============================================================
[2026-05-12 16:24:37][INFO] Temporary directories initialized
[2026-05-12 16:24:37][INFO] Core engine initialized successfully


# 📁 4. Data Manager & Backend Logic

In [4]:
# ==============================================================================
# DATA MANAGER & BACKEND LOGIC
# Batch Processing + Resume System + Image Validation
# Optimized for Kaggle / Colab Runtime
# ==============================================================================

import gc
import json
import os
import re
import shutil
import time
import zipfile
from collections import Counter
from pathlib import Path

import cv2
import gdown
import gradio as gr
from PIL import Image


logger.info("=" * 60)
logger.info("INITIALIZING DATA MANAGER & BACKEND LOGIC")
logger.info("=" * 60)


# ==============================================================================
# DIRECTORY CONFIGURATION
# ==============================================================================

INPUT_DIR = Path("gradio_inputs")
OUTPUT_DIR = Path("gradio_outputs")

INPUT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

ALLOWED_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
    ".bmp",
    ".tiff",
}

PROCESSED_DB = "processed_files.json"


# ==============================================================================
# RESUME PROCESSING SYSTEM
# ==============================================================================

def load_processed_files() -> set:

    if not os.path.exists(PROCESSED_DB):
        return set()

    try:

        with open(PROCESSED_DB, "r") as file:
            return set(json.load(file))

    except Exception:

        logger.exception(
            "Failed to load processed files database"
        )

        return set()


def save_processed_file(filename: str):

    try:

        processed = load_processed_files()

        processed.add(filename)

        with open(PROCESSED_DB, "w") as file:
            json.dump(
                list(processed),
                file,
                indent=2
            )

    except Exception:

        logger.exception(
            f"Failed to save processed file: {filename}"
        )


# ==============================================================================
# DIRECTORY CLEANUP
# ==============================================================================

def clear_folder(folder_path: Path):

    folder_path = Path(folder_path)

    if not folder_path.exists():
        return

    logger.info(f"Clearing folder: {folder_path}")

    for item in folder_path.iterdir():

        try:

            if item.is_file():
                item.unlink()

        except Exception:

            logger.exception(
                f"Failed to delete file: {item}"
            )


# ==============================================================================
# IMAGE VALIDATION
# ==============================================================================

def clean_and_verify_images(folder: Path):

    accepted_stats = Counter()
    rejected_stats = Counter()

    logger.info(
        f"Scanning and validating images: {folder}"
    )

    for path in folder.rglob("*"):

        if not path.is_file():
            continue

        extension = path.suffix.lower() or "no_ext"

        try:

            with Image.open(path) as image:

                image_format = image.format

                image.verify()

            normalized_extension = (
                f".{image_format.lower()}"
                if image_format
                else extension
            )

            if normalized_extension not in ALLOWED_EXTENSIONS:
                raise ValueError("Unsupported image format")

            # ------------------------------------------------------------------
            # AUTO EXTENSION FIX
            # ------------------------------------------------------------------

            if (
                extension not in ALLOWED_EXTENSIONS
                or extension != normalized_extension
            ):

                new_path = path.with_suffix(
                    normalized_extension
                )

                path.rename(new_path)

                logger.info(
                    f"Fixed extension: "
                    f"{path.name} → {new_path.name}"
                )

                extension = normalized_extension

            accepted_stats[extension] += 1

        except Exception:

            rejected_stats[extension] += 1

            logger.warning(
                f"Rejected invalid image: {path.name}"
            )

            try:
                path.unlink()

            except Exception:

                logger.exception(
                    f"Failed to remove invalid image: {path}"
                )

    logger.info(
        f"Validation completed | "
        f"Accepted: {sum(accepted_stats.values())} | "
        f"Rejected: {sum(rejected_stats.values())}"
    )

    return accepted_stats, rejected_stats


# ==============================================================================
# GALLERY HELPER
# ==============================================================================

def get_gallery_images(folder=INPUT_DIR):

    return sorted([
        str(path)
        for path in folder.iterdir()
        if path.suffix.lower() in ALLOWED_EXTENSIONS
    ])


# ==============================================================================
# FILE UPLOAD HANDLER
# ==============================================================================

def handle_upload(files):

    if not files:

        logger.warning("Upload attempted with empty file list")

        return (
            "❌ No files uploaded.",
            get_gallery_images(INPUT_DIR)
        )

    INPUT_DIR.mkdir(exist_ok=True)

    logger.info(f"Uploading {len(files)} file(s)")

    for file_object in files:

        try:

            file_path = Path(file_object.name)

            target_path = INPUT_DIR / file_path.name

            shutil.copy(
                str(file_path),
                str(target_path)
            )

            # --------------------------------------------------------------
            # ZIP EXTRACTION
            # --------------------------------------------------------------

            if zipfile.is_zipfile(target_path):

                logger.info(
                    f"Extracting ZIP archive: "
                    f"{target_path.name}"
                )

                shutil.unpack_archive(
                    str(target_path),
                    str(INPUT_DIR)
                )

                target_path.unlink()

        except Exception:

            logger.exception(
                f"Failed to process uploaded file: "
                f"{file_object.name}"
            )

    accepted, rejected = clean_and_verify_images(INPUT_DIR)

    return (
        f"✅ Upload successful! "
        f"{sum(accepted.values())} valid image(s) loaded.",
        get_gallery_images(INPUT_DIR)
    )


# ==============================================================================
# KAGGLE PATH HANDLER
# ==============================================================================

def handle_kaggle_path(path_string):

    source_path = Path(path_string.strip())

    if not source_path.exists():

        logger.error(
            f"Kaggle path not found: {source_path}"
        )

        return (
            f"❌ Path not found: {source_path}",
            get_gallery_images(INPUT_DIR)
        )

    logger.info(f"Loading Kaggle path: {source_path}")

    INPUT_DIR.mkdir(exist_ok=True)

    try:

        if source_path.is_file():

            shutil.copy(
                str(source_path),
                str(INPUT_DIR / source_path.name)
            )

        else:

            for file in source_path.iterdir():

                if (
                    file.is_file()
                    and file.suffix.lower()
                    in ALLOWED_EXTENSIONS
                ):

                    shutil.copy(
                        str(file),
                        str(INPUT_DIR / file.name)
                    )

        accepted, rejected = clean_and_verify_images(INPUT_DIR)

        return (
            f"✅ Successfully loaded "
            f"{sum(accepted.values())} image(s).",
            get_gallery_images(INPUT_DIR)
        )

    except Exception:

        logger.exception(
            "Failed to load Kaggle path"
        )

        return (
            "❌ Failed to load Kaggle path.",
            get_gallery_images(INPUT_DIR)
        )


# ==============================================================================
# GOOGLE DRIVE HANDLER
# ==============================================================================

def handle_gdrive(url):

    url = url.strip()

    if not url:

        logger.warning("Empty Google Drive URL")

        return (
            "❌ URL is empty.",
            get_gallery_images(INPUT_DIR)
        )

    INPUT_DIR.mkdir(exist_ok=True)

    try:

        logger.info("Starting Google Drive download")

        if "folders/" in url:

            folder_match = re.search(
                r"folders/([a-zA-Z0-9_-]+)",
                url
            )

            if folder_match:

                gdown.download_folder(
                    id=folder_match.group(1),
                    output=str(INPUT_DIR),
                    quiet=True
                )

        else:

            file_match = re.search(
                r"(?:d/|id=)([a-zA-Z0-9_-]+)",
                url
            )

            temp_file = INPUT_DIR / "temp_gdrive_file"

            if file_match:

                gdown.download(
                    id=file_match.group(1),
                    output=str(temp_file),
                    quiet=True
                )

            else:

                gdown.download(
                    url,
                    output=str(temp_file),
                    quiet=True,
                    fuzzy=True
                )

            if (
                temp_file.exists()
                and zipfile.is_zipfile(temp_file)
            ):

                logger.info(
                    "Extracting Google Drive ZIP archive"
                )

                shutil.unpack_archive(
                    temp_file,
                    str(INPUT_DIR)
                )

                temp_file.unlink()

        accepted, rejected = clean_and_verify_images(INPUT_DIR)

        return (
            f"✅ Google Drive import successful! "
            f"{sum(accepted.values())} valid image(s).",
            get_gallery_images(INPUT_DIR)
        )

    except Exception:

        logger.exception(
            "Google Drive import failed"
        )

        return (
            "❌ Google Drive import failed.",
            get_gallery_images(INPUT_DIR)
        )


# ==============================================================================
# RESET HANDLERS
# ==============================================================================

def handle_reset_input():

    clear_folder(INPUT_DIR)

    logger.info("Input folder cleared")

    return (
        "🗑️ Input folder cleared.",
        []
    )


def handle_reset_output():

    clear_folder(OUTPUT_DIR)

    logger.info("Output folder cleared")

    return (
        "🗑️ Output folder cleared.",
        []
    )


# ==============================================================================
# PNG PREPROCESSING
# ==============================================================================

def preprocess_png_if_needed(path: Path):

    if path.suffix.lower() != ".png":
        return path

    try:

        with Image.open(path) as image:

            if image.mode in ("RGBA", "LA", "P"):

                rgb_image = image.convert("RGB")

                rgb_image.save(path)

                logger.info(
                    f"Converted PNG to RGB: {path.name}"
                )

    except Exception:

        logger.exception(
            f"PNG preprocessing failed: {path.name}"
        )

    return path


# ==============================================================================
# BATCH UPSCALE ENGINE
# ==============================================================================

def run_batch_upscale(
    model_type,
    scale,
    quality,
    auto_zip_enabled,
    auto_download_enabled,
    progress=gr.Progress()
):

    if (
        not INPUT_DIR.exists()
        or not list(INPUT_DIR.glob("*"))
    ):

        logger.warning(
            "Batch processing aborted: no input images"
        )

        return (
            "⛔ STOP: No images found in input folder.",
            get_gallery_images(OUTPUT_DIR)
        )

    OUTPUT_DIR.mkdir(exist_ok=True)

    processed_files = load_processed_files()

    files = [

        file for file in INPUT_DIR.iterdir()

        if (
            file.suffix.lower()
            in ALLOWED_EXTENSIONS
            and file.name not in processed_files
        )
    ]

    total_files = len(files)

    success_count = 0
    failed_count = 0

    failed_files = []

    logger.info("=" * 60)
    logger.info(
        f"STARTING BATCH UPSCALE | "
        f"Total Files: {total_files}"
    )
    logger.info("=" * 60)

    start_batch_time = time.time()

    message_log = (
        f"🚀 Starting upscale for "
        f"{total_files} image(s)\n"
    )

    message_log += (
        f"⚙️ Mode: {model_type} | "
        f"Scale: {scale}x | "
        f"Quality: {quality}\n\n"
    )

    for index, file_path in enumerate(

        progress.tqdm(
            files,
            desc="Upscaling Images"
        )

    ):

        image_start_time = time.time()

        logger.info(
            f"[{index + 1}/{total_files}] "
            f"Processing: {file_path.name}"
        )

        try:

            clean_path = preprocess_png_if_needed(
                file_path
            )

            result_path = upscale_image(
                image_path=str(clean_path),
                output_folder=str(OUTPUT_DIR),

                image_type=(
                    "anime"
                    if model_type == "Anime / Cartoon"
                    else "general"
                ),

                scale=int(scale),
                quality=quality.lower()
            )

            if result_path and Path(result_path).exists():

                success_count += 1

                save_processed_file(file_path.name)

                elapsed = (
                    time.time() - image_start_time
                )

                logger.info(
                    f"Processing successful: "
                    f"{file_path.name} "
                    f"({elapsed:.2f}s)"
                )

            else:

                failed_count += 1

                failed_files.append(file_path.name)

                logger.error(
                    f"Processing failed: {file_path.name}"
                )

        except Exception:

            failed_count += 1

            failed_files.append(file_path.name)

            logger.exception(
                f"Fatal batch error: {file_path.name}"
            )

        finally:

            # --------------------------------------------------------------
            # MEMORY CLEANUP
            # --------------------------------------------------------------

            try:

                gc.collect()

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            except Exception:

                logger.exception(
                    "Batch memory cleanup failed"
                )

    total_runtime = (
        time.time() - start_batch_time
    )

    logger.info("=" * 60)
    logger.info("BATCH UPSCALE COMPLETED")
    logger.info("=" * 60)

    logger.info(f"Success Count : {success_count}")
    logger.info(f"Failed Count  : {failed_count}")
    logger.info(f"Runtime       : {total_runtime:.2f}s")

    if failed_files:

        logger.warning(
            f"Failed Files: {', '.join(failed_files)}"
        )

    message_log += (
        f"\n✅ FINISHED!\n"
        f"Success: {success_count}\n"
        f"Failed: {failed_count}\n"
        f"Runtime: {total_runtime:.2f}s"
    )

    download_update = gr.update(
        visible=False,
        value=None
    )
    
    if auto_zip_enabled:
    
        logger.info(
            "Auto ZIP creation enabled"
        )
    
        try:
    
            zip_filename = "Real_ESRGAN_Results"
    
            zip_path = Path(f"{zip_filename}.zip")
    
            if zip_path.exists():
                zip_path.unlink()
    
            shutil.make_archive(
                zip_filename,
                "zip",
                str(OUTPUT_DIR)
            )
    
            logger.info(
                "ZIP archive created successfully"
            )
    
            message_log += (
                "\n📦 ZIP archive created successfully."
            )
    
            if auto_download_enabled:
    
                download_update = gr.update(
                    value=str(zip_path),
                    visible=True
                )
    
        except Exception:
    
            logger.exception(
                "Automatic ZIP creation failed"
            )
    
            message_log += (
                "\n❌ Failed to create ZIP archive."
            )
    
    return (
        message_log,
        get_gallery_images(OUTPUT_DIR),
        download_update
    )


logger.info(
    "Data manager and backend logic initialized successfully"
)

[2026-05-12 16:24:44][INFO] ============================================================
[2026-05-12 16:24:44][INFO] INITIALIZING DATA MANAGER & BACKEND LOGIC
[2026-05-12 16:24:44][INFO] ============================================================
[2026-05-12 16:24:44][INFO] Data manager and backend logic initialized successfully


# 🎨 5. Gradio UI / UX Interface

In [5]:
# ==============================================================================
# GRADIO UI / UX INTERFACE
# Professional Notebook Interface for Easy-RealESRGAN
# Optimized for Kaggle / Colab Runtime
# ==============================================================================

import shutil
from pathlib import Path

import gradio as gr


logger.info("=" * 60)
logger.info("BUILDING GRADIO USER INTERFACE")
logger.info("=" * 60)


# ==============================================================================
# CUSTOM CSS
# ==============================================================================

CUSTOM_CSS = """
/* =========================================================
   FORCE STABLE GALLERY GRID
========================================================= */

.strict-gallery .grid-container {
    grid-template-columns: repeat(4, minmax(0, 1fr)) !important;
}

.strict-gallery-out .grid-container {
    grid-template-columns: repeat(3, minmax(0, 1fr)) !important;
}


/* =========================================================
   HIDE TEXTBOX SCROLLBAR
========================================================= */

.hide-scrollbar textarea {
    overflow-y: hidden !important;
}


/* =========================================================
   SCROLLABLE FILE PREVIEW
========================================================= */

.scrollable-file .file-preview {
    max-height: 220px !important;
    overflow-y: auto !important;
}
"""


# ==============================================================================
# GALLERY SELECTION HANDLER
# ==============================================================================

def on_select_gallery(evt: gr.SelectData):

    try:

        images = get_gallery_images(INPUT_DIR)

        if evt.index < len(images):

            file_path = images[evt.index]

            logger.info(
                f"Gallery image selected: "
                f"{Path(file_path).name}"
            )

            return (
                file_path,
                f"Selected: {Path(file_path).name}",
                gr.update(interactive=True)
            )

    except Exception:

        logger.exception(
            "Gallery selection failed"
        )

    return (
        None,
        "",
        gr.update(interactive=False)
    )


# ==============================================================================
# SINGLE FILE DELETE HANDLER
# ==============================================================================

def on_delete_single(file_path):

    try:

        if file_path and Path(file_path).exists():

            Path(file_path).unlink()

            logger.info(
                f"Deleted input image: "
                f"{Path(file_path).name}"
            )

            return (
                f"🗑️ {Path(file_path).name} deleted successfully.",
                get_gallery_images(INPUT_DIR),
                None,
                "",
                gr.update(interactive=False)
            )

    except Exception:

        logger.exception(
            "Failed to delete selected image"
        )

    return (
        "❌ Failed to delete selected image.",
        get_gallery_images(INPUT_DIR),
        None,
        "",
        gr.update(interactive=False)
    )


# ==============================================================================
# ZIP EXPORT HANDLER
# ==============================================================================

def handle_create_zip(progress=gr.Progress()):

    output_dir = Path("gradio_outputs")

    try:

        if (
            not output_dir.exists()
            or not list(output_dir.iterdir())
        ):

            logger.warning(
                "ZIP creation failed: output folder is empty"
            )

            return (
                "❌ Output folder is empty. Nothing to ZIP.",
                gr.update(
                    value=None,
                    visible=False
                )
            )

        progress(0, desc="Creating ZIP archive...")

        zip_filename = "Real_ESRGAN_Results"

        zip_path = Path(f"{zip_filename}.zip")

        if zip_path.exists():
            zip_path.unlink()

        logger.info("Creating ZIP archive")

        shutil.make_archive(
            zip_filename,
            "zip",
            str(output_dir)
        )

        progress(1.0, desc="ZIP archive ready!")

        logger.info(
            f"ZIP archive created successfully: {zip_path}"
        )

        return (
            "✅ ZIP archive created successfully!",
            gr.update(
                value=str(zip_path),
                visible=True
            )
        )

    except Exception:

        logger.exception(
            "ZIP archive creation failed"
        )

        return (
            "❌ Failed to create ZIP archive.",
            gr.update(
                value=None,
                visible=False
            )
        )


# ==============================================================================
# MAIN UI BUILDER
# ==============================================================================

def build_ui():

    logger.info("Initializing Gradio interface")

    theme = gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="slate"
    )

    with gr.Blocks(
        theme=theme,
        title="Easy-RealESRGAN",
        css=CUSTOM_CSS
    ) as demo:

        # ----------------------------------------------------------------------
        # HEADER
        # ----------------------------------------------------------------------

        gr.Markdown(
            "# 🚀 Easy-RealESRGAN Image Upscaler"
        )

        gr.Markdown(
            "Professional Open-Source AI Upscaling "
            "Notebook for Kaggle & Colab"
        )

        with gr.Tabs():

            # ==================================================================
            # TAB 1 — DATA MANAGER
            # ==================================================================

            with gr.Tab("📁 Data Manager"):

                with gr.Row():

                    # ----------------------------------------------------------
                    # LEFT PANEL
                    # ----------------------------------------------------------

                    with gr.Column(scale=1):

                        # ------------------------------------------------------
                        # FILE UPLOAD
                        # ------------------------------------------------------

                        with gr.Accordion(
                            "Upload Images / ZIP",
                            open=True
                        ):

                            upload_component = gr.File(
                                file_count="multiple",
                                label="Drop Image or ZIP Files",
                                elem_classes="scrollable-file"
                            )

                            button_upload = gr.Button(
                                "Upload & Extract",
                                variant="primary"
                            )

                        # ------------------------------------------------------
                        # KAGGLE DATASET
                        # ------------------------------------------------------

                        with gr.Accordion(
                            "Load from Kaggle Dataset",
                            open=False
                        ):

                            kaggle_path_component = gr.Textbox(
                                placeholder="/kaggle/input/...",
                                label="Kaggle Dataset Path"
                            )

                            button_kaggle = gr.Button(
                                "Load Dataset",
                                variant="primary"
                            )

                        # ------------------------------------------------------
                        # GOOGLE DRIVE
                        # ------------------------------------------------------

                        with gr.Accordion(
                            "Import from Google Drive",
                            open=False
                        ):

                            gdrive_component = gr.Textbox(
                                placeholder="https://drive.google.com/...",
                                label="Google Drive URL"
                            )

                            button_gdrive = gr.Button(
                                "Import from GDrive",
                                variant="primary"
                            )

                        # ------------------------------------------------------
                        # STATUS BOX
                        # ------------------------------------------------------

                        status_text_input = gr.Textbox(
                            label="Status",
                            interactive=False,
                            lines=1,
                            max_lines=4,
                            elem_classes="hide-scrollbar"
                        )

                        button_reset_input = gr.Button(
                            "🗑️ Clear Input Folder",
                            variant="stop"
                        )

                    # ----------------------------------------------------------
                    # RIGHT PANEL
                    # ----------------------------------------------------------

                    with gr.Column(scale=2):

                        gr.Markdown(
                            "### 🖼️ Input Gallery"
                        )

                        gallery_input = gr.Gallery(
                            show_label=False,
                            columns=4,
                            rows=2,
                            height=450,
                            object_fit="contain",
                            allow_preview=True,
                            elem_classes="strict-gallery"
                        )

                        with gr.Row():

                            selected_file_state = gr.State(
                                None
                            )

                            selected_file_text = gr.Textbox(
                                show_label=False,
                                placeholder=(
                                    "Select an image to delete..."
                                ),
                                interactive=False,
                                scale=4,
                                lines=2,
                                max_lines=5,
                                elem_classes="hide-scrollbar"
                            )

                            button_delete_single = gr.Button(
                                "🗑️ Delete",
                                variant="stop",
                                interactive=False,
                                scale=1
                            )

            # ==================================================================
            # TAB 2 — PLAYGROUND
            # ==================================================================

            with gr.Tab("⚙️ Batch Upscale Playground"):

                with gr.Row():

                    # ----------------------------------------------------------
                    # LEFT PANEL
                    # ----------------------------------------------------------

                    with gr.Column(scale=1):

                        gr.Markdown(
                            "### 🎛️ Upscale Configuration"
                        )

                        model_type = gr.Radio(
                            [
                                "General / Photo",
                                "Anime / Cartoon"
                            ],
                            value="General / Photo",
                            label="Model Type"
                        )

                        scale = gr.Radio(
                            ["2", "4"],
                            value="4",
                            label="Scale Factor"
                        )

                        quality = gr.Dropdown(
                            choices=[
                                "ultra",
                                "high",
                                "balanced",
                                "fast"
                            ],

                            value="balanced",

                            label="Quality Preset",

                            info=(
                                "Ultra = Highest Quality / "
                                "Fast = Lowest VRAM Usage"
                            )
                        )

                        auto_zip = gr.Checkbox(
                            value=True,
                            label="Auto Create ZIP After Processing"
                        )
                        
                        auto_download = gr.Checkbox(
                            value=False,
                            label="Auto Show Download After Processing"
                        )

                        button_run_batch = gr.Button(
                            "🚀 START BATCH UPSCALE",
                            variant="primary",
                            size="lg"
                        )

                        status_text_output = gr.Textbox(
                            label="Processing Log",
                            interactive=False,
                            lines=2,
                            max_lines=10,
                            elem_classes="hide-scrollbar"
                        )

                        with gr.Row():

                            button_create_zip = gr.Button(
                                "📦 Create ZIP",
                                variant="secondary"
                            )

                            button_reset_output = gr.Button(
                                "🗑️ Clear Output",
                                variant="stop"
                            )

                        download_zip = gr.File(
                            label="Download Results (ZIP)",
                            interactive=False,
                            visible=False
                        )

                    # ----------------------------------------------------------
                    # RIGHT PANEL
                    # ----------------------------------------------------------

                    with gr.Column(scale=2):

                        gr.Markdown(
                            "### ✨ Output Gallery"
                        )

                        gallery_output = gr.Gallery(
                            show_label=False,
                            columns=3,
                            rows=2,
                            height=450,
                            object_fit="contain",
                            allow_preview=True,
                            elem_classes="strict-gallery-out"
                        )

        # ======================================================================
        # EVENT LISTENERS
        # ======================================================================

        logger.info("Registering Gradio event listeners")

        # ----------------------------------------------------------------------
        # INPUT HANDLERS
        # ----------------------------------------------------------------------

        button_upload.click(
            handle_upload,
            inputs=[upload_component],
            outputs=[
                status_text_input,
                gallery_input
            ]
        )

        button_kaggle.click(
            handle_kaggle_path,
            inputs=[kaggle_path_component],
            outputs=[
                status_text_input,
                gallery_input
            ]
        )

        button_gdrive.click(
            handle_gdrive,
            inputs=[gdrive_component],
            outputs=[
                status_text_input,
                gallery_input
            ]
        )

        button_reset_input.click(
            handle_reset_input,
            outputs=[
                status_text_input,
                gallery_input
            ]
        )

        # ----------------------------------------------------------------------
        # GALLERY EVENTS
        # ----------------------------------------------------------------------

        gallery_input.select(
            on_select_gallery,
            inputs=None,
            outputs=[
                selected_file_state,
                selected_file_text,
                button_delete_single
            ]
        )

        button_delete_single.click(
            on_delete_single,
            inputs=[selected_file_state],
            outputs=[
                status_text_input,
                gallery_input,
                selected_file_state,
                selected_file_text,
                button_delete_single
            ]
        )

        # ----------------------------------------------------------------------
        # UPSCALE EVENTS
        # ----------------------------------------------------------------------

        button_run_batch.click(
            run_batch_upscale,
            inputs=[
                model_type,
                scale,
                quality,
                auto_zip,
                auto_download
            ],
            outputs=[
                status_text_output,
                gallery_output
            ]
        )

        button_reset_output.click(
            handle_reset_output,
            outputs=[
                status_text_output,
                gallery_output,
                download_zip
            ]
        )

        # ----------------------------------------------------------------------
        # ZIP EXPORT EVENT
        # ----------------------------------------------------------------------

        button_create_zip.click(
            handle_create_zip,
            outputs=[
                status_text_output,
                download_zip
            ]
        )

    logger.info(
        "Gradio interface built successfully"
    )

    return demo


logger.info(
    "UI/UX system initialized successfully"
)

[2026-05-12 16:24:44][INFO] ============================================================
[2026-05-12 16:24:44][INFO] BUILDING GRADIO USER INTERFACE
[2026-05-12 16:24:44][INFO] ============================================================
[2026-05-12 16:24:44][INFO] UI/UX system initialized successfully


# 🌐 6. Cloudflare Tunnel Launcher

In [6]:
# ==============================================================================
# CLOUDFLARE TUNNEL & APPLICATION LAUNCHER
# Public Gradio Access for Kaggle / Colab Runtime
# Optimized for Stability & Open-Source Usage
# ==============================================================================

import os
import queue
import re
import socket
import subprocess
import threading
import urllib.request
from pathlib import Path
from IPython.display import HTML, display


logger.info("=" * 60)
logger.info("INITIALIZING CLOUDFLARE TUNNEL SYSTEM")
logger.info("=" * 60)


# ==============================================================================
# CLOUDFLARED CONFIGURATION
# ==============================================================================

CLOUDFLARED_BINARY = Path("cloudflared")

CLOUDFLARED_URL = (
    "https://github.com/cloudflare/cloudflared/"
    "releases/latest/download/cloudflared-linux-amd64"
)


# ==============================================================================
# PORT MANAGEMENT
# ==============================================================================

def get_free_port() -> int:
    """
    Find an available local port dynamically.
    """

    try:

        with socket.socket(
            socket.AF_INET,
            socket.SOCK_STREAM
        ) as socket_instance:

            socket_instance.bind(("", 0))

            port = socket_instance.getsockname()[1]

            logger.info(
                f"Dynamic free port allocated: {port}"
            )

            return port

    except Exception:

        logger.exception(
            "Failed to allocate free port"
        )

        raise


# ==============================================================================
# CLOUDFLARED INSTALLER
# ==============================================================================

def ensure_cloudflared_installed():
    """
    Download cloudflared binary if not already installed.
    """

    try:

        if CLOUDFLARED_BINARY.exists():

            logger.info(
                "Cloudflared binary already exists"
            )

            return

        logger.info(
            "Downloading cloudflared binary..."
        )

        urllib.request.urlretrieve(
            CLOUDFLARED_URL,
            str(CLOUDFLARED_BINARY)
        )

        os.chmod(CLOUDFLARED_BINARY, 0o777)

        logger.info(
            "Cloudflared installed successfully"
        )

    except Exception:

        logger.exception(
            "Failed to install cloudflared"
        )

        raise


# ==============================================================================
# CLOUDFLARE TUNNEL STARTER
# ==============================================================================

def start_cloudflared(
    port: int,
    url_queue: queue.Queue
):
    """
    Start Cloudflare Tunnel in background thread
    and push public URL into queue.
    """

    logger.info(
        "Starting Cloudflare Tunnel..."
    )

    cmd = [
        f"./{CLOUDFLARED_BINARY.name}",
        "tunnel",
        "--url",
        f"http://127.0.0.1:{port}"
    ]

    try:

        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )

        url_found = False

        for line in process.stdout:

            clean_line = line.strip()

            IMPORTANT_KEYWORDS = [
                "Registered tunnel",
                "ERR",
            ]
            
            if any(
                keyword in clean_line
                for keyword in IMPORTANT_KEYWORDS
            ):
            
                logger.debug(
                    f"[Cloudflared] {clean_line}"
                )

            if not url_found:

                match = re.search(
                    r"(https://[^\s]+trycloudflare\.com)",
                    line
                )
                
                if match:
                
                    cloudflare_url = match.group(1).strip()
                
                    url_queue.put(cloudflare_url)
                
                    url_found = True

    except Exception:

        logger.exception(
            "Cloudflare tunnel failed to start"
        )

        url_queue.put(None)


# ==============================================================================
# APPLICATION LAUNCHER
# ==============================================================================

def launch_application():

    try:

        logger.info(
            "Launching Easy-RealESRGAN application"
        )

        ensure_cloudflared_installed()

        free_port = get_free_port()

        logger.info(
            f"Selected application port: {free_port}"
        )

        # ------------------------------------------------------------------
        # BUILD UI
        # ------------------------------------------------------------------

        logger.info(
            "Building Gradio interface..."
        )

        ui = build_ui()

        # ------------------------------------------------------------------
        # START GRADIO FIRST
        # ------------------------------------------------------------------

        logger.info(
            "Starting Gradio server..."
        )

        logger.info(
            f"Launching Gradio on port {free_port}"
        )
        
        ui.launch(
            server_name="127.0.0.1",
            server_port=free_port,
            share=False,
            inline=False,
            quiet=True
        )

        # ------------------------------------------------------------------
        # WAIT SERVER INITIALIZATION
        # ------------------------------------------------------------------

        import time

        logger.info(
            "Waiting for Gradio server initialization..."
        )

        time.sleep(5)

        # ------------------------------------------------------------------
        # START CLOUDFLARE AFTER SERVER READY
        # ------------------------------------------------------------------

        url_queue = queue.Queue()

        tunnel_thread = threading.Thread(
            target=start_cloudflared,
            args=(free_port, url_queue),
            daemon=True
        )

        tunnel_thread.start()

        logger.info(
            "Waiting for Cloudflare public URL..."
        )

        try:

            cloudflare_url = url_queue.get(timeout=30)
        
        except queue.Empty:
        
            logger.error(
                "Timed out while waiting for Cloudflare URL"
            )
        
            return

        if not cloudflare_url:

            logger.error(
                "Failed to retrieve Cloudflare URL"
            )

            return

        # ------------------------------------------------------------------
        # DISPLAY PUBLIC URL
        # ------------------------------------------------------------------

        logger.info("=" * 60)
        logger.info("APPLICATION IS READY")
        logger.info("=" * 60)

        display(

            HTML(f"""
        
            <div style="
                padding: 18px;
                border-radius: 12px;
                background: linear-gradient(135deg, #1e3c72, #2a5298);
                color: white;
                font-family: Arial;
                margin: 12px 0;
            ">
        
                <h2 style="margin-top:0;">
                    🚀 Easy-RealESRGAN is Online!
                </h2>
        
                <p>
                    Click the link below to open the application:
                </p>
        
                <a
                    href="{cloudflare_url}"
                    target="_blank"
                    style="
                        color: #FFD700;
                        font-size: 18px;
                        font-weight: bold;
                        text-decoration: none;
                        word-break: break-all;
                    "
                >
                    {cloudflare_url}
                </a>
        
            </div>
        
            """)
        
        )

        logger.info(
            "Application launched successfully"
        )

    except Exception:

        logger.exception(
            "Application launcher failed"
        )


# ==============================================================================
# EXECUTION
# ==============================================================================

if __name__ == "__main__":

    launch_application()

[2026-05-12 16:24:44][INFO] ============================================================
[2026-05-12 16:24:44][INFO] INITIALIZING CLOUDFLARE TUNNEL SYSTEM
[2026-05-12 16:24:44][INFO] ============================================================
[2026-05-12 16:24:44][INFO] Launching Easy-RealESRGAN application
[2026-05-12 16:24:44][INFO] Downloading cloudflared binary...
[2026-05-12 16:24:45][INFO] Cloudflared installed successfully
[2026-05-12 16:24:45][INFO] Dynamic free port allocated: 40763
[2026-05-12 16:24:45][INFO] Selected application port: 40763
[2026-05-12 16:24:45][INFO] Building Gradio interface...
[2026-05-12 16:24:45][INFO] Initializing Gradio interface
[2026-05-12 16:24:45][INFO] Registering Gradio event listeners
[2026-05-12 16:24:45][INFO] Gradio interface built successfully
[2026-05-12 16:24:45][INFO] Starting Gradio server...
[2026-05-12 16:24:45][INFO] Launching Gradio on port 40763
[2026-05-12 16:24:45][INFO] Waiting for Gradio server initialization...
[2026-05-12 16

[2026-05-12 16:24:54][INFO] Application launched successfully


---

# ☕ Support the Project

If this project has been helpful to you, please consider supporting its development.  
Your support helps maintain and improve the project, fund future research, and continue building better open-source AI tools for the community.

Every contribution truly means a lot and helps push this project further. 🚀

| Platform | Link |
| :--- | :--- |
| **Trakteer** | https://trakteer.id/pyforge |
| **Saweria** | https://saweria.co/pyforge |
| **SociaBuzz** | https://sociabuzz.com/pyforge |
| **PayPal** | https://www.paypal.com/paypalme/Masyura |

---

# ⚖️ License

This project is licensed under the **MIT License**.

You are free to use, modify, and distribute this project, provided that the original copyright and license notice are included.

Copyright (c) 2026 **MasyuraC7**

---